# Registration

<b>Steps</b>:
1. Tissue annotation was performed on the original unwarped images in Qupath
2. Transcripts mask was created (see code below)
3. Registration was performed in BigWarp (see detailed explanation below)
4. Tissue annotations were warped according to the same registration (see code below)

<b>Tools used</b>:
- QuPath v0.5.1
- Fiji (ImageJ 1.54p)
- Fiji/plugins/bigwarp_fiji-9.1.3.jar + Fiji/jars/bigdataviewer_fiji-6.3.0.jar

<b> Bigwarp registration</b>:
- Started Fiji and opened microscopy image and transcript mask
- Plugins > BigDataViewer > Big Warp Command
    - Added both the micropscopy image and the transcript mask.
    - Transcript image was used as fixed image and the microscopy image was set to moving image.
- Landmarks were placed on the fiducial concentric circles visible in both the moving image and the fixed image
- `Thin Plate Spline` was used as transformation type resulting in an elastic transformation to warp the moving image.
- Warped micropscopy image was exported (`warped.tif`) together with the landmarks (`landmarks.csv`).

<b> Create transcript mask</b>:

In [7]:
import harpy as hp
import spatialdata as sd
import numpy as np
import math
import dask.array as da
from skimage import io

def transcripts_range(sdata: sd.SpatialData) -> tuple[np.int32, np.int32, np.int32, np.int32]:
    transcripts = sdata.points["transcripts"]
    xs = transcripts["x"]
    ys = transcripts["y"]
    assert xs.dtype == np.float64
    assert ys.dtype == np.float64
    xmin = xs.min().compute()
    xmax = xs.max().compute()
    ymin = ys.min().compute()
    ymax = ys.max().compute()
    return math.floor(xmin), math.ceil(xmax), math.floor(ymin), math.ceil(ymax)

def make_transcripts_mask(sdata: sd.SpatialData) -> np.ndarray:
    xmin, xmax, ymin, ymax = transcripts_range(sdata)
    assert xmin == 0
    assert ymin == 0
    
    transcripts = sdata.points["transcripts"]
    xs = transcripts["x"]
    ys = transcripts["y"]

    xs_np = np.rint(xs.compute().to_numpy()).astype(int)
    ys_np = np.rint(ys.compute().to_numpy()).astype(int)

    mask = np.zeros((ymax + 1, xmax + 1), dtype="uint8")
    mask[ys_np, xs_np] = 255
    return mask

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [8]:
# Specify samples
samples = [
    'Exp65-sampleSPC014',
    'Exp93-sampleSPC002',
    'Exp93-sampleSPC022',
    'Exp100-samplePSB',
    'Exp100-sampleSPC004'
]

In [9]:
for sample in samples:
    # Paths
    path_transcripts = f'e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/{sample}/NovaScope/novascope_output/analysis/{sample}/{sample}-default/preprocess/{sample}-default.transcripts.tsv.gz'
    output_path = f'E:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/{sample}/BigWarp/transcript_mask.tiff'

    # Create empty SpatialData
    sdata = sd.SpatialData()

    # Read transcripts
    sdata = hp.io.read_transcripts(
        sdata,
        path_count_matrix=path_transcripts,
        column_x=2,
        column_y=3,
        column_z=None,
        column_gene=5,
        column_midcount=6,
        delimiter='\t',
        header=0,
        comment=None,
        transform_matrix=np.array([[0.001, 0, 0],[0, 0.001, 0],[0, 0, 1]]),
        overwrite=True
    )

    # Create transcript mask
    transcripts_mask = make_transcripts_mask(sdata)

    sdata = hp.im.add_image_layer(
        sdata,
        arr = da.from_array(transcripts_mask[np.newaxis, :, :]),
        dims = ("c", "y", "x"),
        output_layer = "transcripts_mask",
        chunks = 1024,
        overwrite = True,
    )

    # Save transcript mask
    io.imsave(output_path, transcripts_mask)

    # Inspect SpatialData
    sdata

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\dask\dataframe\io\csv.py:555: UserWarning: Warning gzip compression does not support breaking apart files
Please ensure that each individual file can fit in memory and
use the keyword ``blocksize=None to remove this message``
Setting ``blocksize=None``
  warn(
2025-12-22 13:42:08,830 - harpy.io._transcripts - INFO - Transform matrix used:
 [[0.001 0.    0.   ]
 [0.    0.001 0.   ]
 [0.    0.    1.   ]]
2025-12-22 13:45:35,649 - harpy.image._manager - INFO - Writing results to layer 'transcripts_mask'
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\dask\dataframe\io\csv.py:555: UserWarning: Warning gzip compression does not support breaking apart files
Please ensure that each individual file can fit in memory and
use the keyword ``blocksize=None to remove this message``
Setting ``blocksize=None``
  warn(
2025-12-22 13:45:36,404 - harpy.io._transcripts - INFO - Transform matrix used:
 [[0.001 0.    0.   ]
 [0.    0.001

<b> Warp tissue annotations </b>:

In [10]:
import os
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from utils.transform_geometries import convert_geometries_for_bigwarp, create_warped_geometries

In [12]:
for sample in samples:
    # Convert GeoJSON into CSV for BigWarp
    input_geojson = f"e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/{sample}/tissue_annotation/tissue.geojson"
    output_csv = f"e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/{sample}/BigWarp/tissue_coordinates.csv"

    convert_geometries_for_bigwarp(input_geojson, output_csv)

Loaded 1 geometries from 'e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp65-sampleSPC014/tissue_annotation/tissue.geojson'.
Saved 47130 coordinate rows from 1 geometries to 'e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp65-sampleSPC014/BigWarp/tissue_coordinates.csv'.
Loaded 1 geometries from 'e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC002/tissue_annotation/tissue.geojson'.
Saved 71314 coordinate rows from 1 geometries to 'e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC002/BigWarp/tissue_coordinates.csv'.
Loaded 1 geometries from 'e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/tissue_annotation/tissue.geojson'.
Saved 38546 coordinate rows from 1 geometries to 'e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/BigWarp/tissue_coordinates.csv'.
Loaded 10 geometries from 'e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp100-samplePSB/tissue_annotation/tissue.geojson'.
Saved 10427 coordinate ro

Run warp_points_bigwarp.groovy script in FIJI for each of the samples. 
- Supply landmark file (`landmarks.csv`), Input coordinates (`tissue_coordinates.csv`) and set path to Output coordinates (`tissue_coordinates_warped.csv`). 
- X and Y columns names are set to `x` and `y`, respectively, Output X and Y column names are set to `x_warped` and `y_warped`, respectively, and Transformation type is `Thin Plate Spline`. 
- Scaling factors are set to 1 and Reverse direction is unchecked.

In [13]:
for sample in samples:
    # Convert warped BigWarp CSV into GeoJSON
    input_csv = f"e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/{sample}/BigWarp/tissue_coordinates_warped.csv"
    output_geojson = f"e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/{sample}/BigWarp/tissue_warped.geojson"
    x_col = "x_warped"
    y_col = "y_warped"

    create_warped_geometries(input_csv, x_col, y_col, output_geojson)

Loaded 47130 rows from 'e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp65-sampleSPC014/BigWarp/tissue_coordinates_warped.csv'.
Saved warped GeoJSON with 1 geometries to 'e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp65-sampleSPC014/BigWarp/tissue_warped.geojson'.
Loaded 71314 rows from 'e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC002/BigWarp/tissue_coordinates_warped.csv'.
Saved warped GeoJSON with 1 geometries to 'e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC002/BigWarp/tissue_warped.geojson'.
Loaded 38546 rows from 'e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/BigWarp/tissue_coordinates_warped.csv'.
Saved warped GeoJSON with 1 geometries to 'e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp93-sampleSPC022/BigWarp/tissue_warped.geojson'.
Loaded 10427 rows from 'e:/2024-07-UTBenchmark-SpC/data/processed/Nova-ST/Exp100-samplePSB/BigWarp/tissue_coordinates_warped.csv'.
Saved warped GeoJSON with 10 geometrie